In [41]:
import numpy as np
import joblib
import random
import time
from scipy.ndimage import rotate, shift, zoom
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, HalvingRandomSearchCV
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import classification_report, accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline  # 파이프라인 추가

In [43]:
# ==========================================
# 1. 사용자 정의 전처리 클래스
# ==========================================
class CustomImagePreprocess(BaseEstimator, TransformerMixin):
    def __init__(self, stroke_target=0.4, target_size=20, final_size=28, noise_threshold=0.02):
        self.stroke_target = stroke_target       
        self.target_size = target_size           
        self.final_size = final_size             
        self.noise_threshold = noise_threshold   
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        X = np.array(X)
        if X.ndim == 2:
            side = int(np.sqrt(X.shape[1]))
            X = X.reshape(-1, side, side)
            
        processed = []
        for img in X:
            img = self.normalize_scale(img)   
            img = self.remove_noise(img)      
            img = self.normalize_stroke(img)  
            img = self.normalize_size(img)    
            img = self.center_align(img)
            processed.append(img)
        
        processed = np.array(processed)
        return processed.reshape(len(processed), -1)
    
    # --- 내부 메서드 ---
    def normalize_scale(self, img):
        img = img.astype(np.float32)
        if img.max() > 1: img = img / 255.0
        return img
    
    def remove_noise(self, img):
        img = img.copy()
        img[img < self.noise_threshold] = 0.0
        return img
    
    def normalize_stroke(self, img):
        mask = img > 0.05
        if mask.sum() == 0: return img
        mean_val = img[mask].mean()
        if mean_val == 0: return img
        scale_factor = self.stroke_target / mean_val
        img = img * scale_factor
        img = np.clip(img, 0, 1)
        return img
           
    def normalize_size(self, img):
        coords = np.where(img > 0.05)
        if coords[0].size == 0: return np.zeros((self.final_size, self.final_size))
        r1, r2 = coords[0].min(), coords[0].max()
        c1, c2 = coords[1].min(), coords[1].max()
        digit = img[r1:r2+1, c1:c2+1]
        h, w = digit.shape
        if h == 0 or w == 0: return np.zeros((self.final_size, self.final_size))
        zoom_h = self.target_size / h
        zoom_w = self.target_size / w
        digit_resized = zoom(digit, (zoom_h, zoom_w))
        return digit_resized
    
    def center_align(self, img):
        canvas = np.zeros((self.final_size, self.final_size))
        h, w = img.shape
        if h > self.final_size or w > self.final_size:
            h_start = (h - self.final_size) // 2
            w_start = (w - self.final_size) // 2
            img = img[h_start:h_start+self.final_size, w_start:w_start+self.final_size]
            h, w = img.shape
        y0 = (self.final_size - h) // 2
        x0 = (self.final_size - w) // 2
        canvas[y0:y0+h, x0:x0+w] = img
        return canvas

In [3]:
# ==========================================
# 2. 데이터 증강 함수 (그대로 유지)
# ==========================================
def apply_augment_single(img, mode, rotate_range=15, shift_range=2, zoom_range=0.1):
    if mode == 'rotate':
        angle = random.uniform(-rotate_range, rotate_range)
        img = rotate(img, angle, reshape=False, mode='constant', cval=0)
    elif mode == 'shift':
        dx = random.uniform(-shift_range, shift_range)
        dy = random.uniform(-shift_range, shift_range)
        img = shift(img, [dy, dx], mode='constant', cval=0)
    elif mode == 'zoom':
        factor = random.uniform(1.0, 1.0 + zoom_range)
        h, w = img.shape
        zoomed_img = zoom(img, factor, mode='constant', cval=0)
        zh, zw = zoomed_img.shape
        start_h = max((zh - h) // 2, 0)
        start_w = max((zw - w) // 2, 0)
        img = zoomed_img[start_h:start_h+h, start_w:start_w+w]
        if img.shape != (28, 28):
            img = img[:28, :28]
    return img

def custom_augment(X, y, multiplier=1):
    if multiplier <= 1:
        return X, y

    X = np.array(X)
    y = np.array(y)
    
    if X.ndim == 2:
        X_reshaped = X.reshape(-1, 28, 28)
    else:
        X_reshaped = X

    X_out = []
    y_out = []

    for img, label in zip(X_reshaped, y):
        X_out.append(img)
        y_out.append(label)
        for _ in range(multiplier - 1):
            mode = random.choice(['rotate', 'shift', 'zoom'])
            aug_img = apply_augment_single(img, mode)
            X_out.append(aug_img)
            y_out.append(label)

    X_out = np.array(X_out).reshape(len(X_out), -1)
    y_out = np.array(y_out)
    return X_out, y_out

In [38]:
# ==========================================
# 3. 데이터 로드 및 전처리 (순서 유지: 전처리 -> 증강)
# ==========================================
def load_preprocess_and_augment(hand_path, org_path, augment_factor=1):
    print(f"\n[Data Prep] Loading & Preprocessing... Augment Factor: {augment_factor}x")
    
    # 1) 데이터 로드
    try:
        d_hand = np.load(hand_path, allow_pickle=True)
        X_hand, y_hand = d_hand['x_train'], d_hand['y_train'].astype(int)
        
        d_org = np.load(org_path, allow_pickle=True)
        X_org, y_org = d_org['x_train'], d_org['y_train'].astype(int)
    except Exception as e:
        print(f"Error loading files: {e}")
        return None, None, None, None

    # 2) 전처리 (먼저 수행)
    preprocessor = CustomImagePreprocess()
    print("  - Preprocessing Handmade data...")
    X_hand_processed = preprocessor.transform(X_hand)
    print("  - Preprocessing Original MNIST data...")
    X_org_processed = preprocessor.transform(X_org)

    # 3) 분할
    X_hand_tr, X_hand_te, y_hand_tr, y_hand_te = train_test_split(
        X_hand_processed, y_hand, test_size=0.2, random_state=42, stratify=y_hand
    )
    X_org_tr, X_org_te, y_org_tr, y_org_te = train_test_split(
        X_org_processed, y_org, test_size=0.2, random_state=42, stratify=y_org
    )

    # 4) 증강 (전처리된 데이터를 바탕으로 증강)
    if augment_factor > 1:
        print(f"  - Augmenting Handmade Train ({len(X_hand_tr)} -> {augment_factor}x)...")
        X_hand_tr, y_hand_tr = custom_augment(X_hand_tr, y_hand_tr, multiplier=augment_factor)
    
    # 5) 병합
    X_train = np.concatenate([X_hand_tr, X_org_tr])
    y_train = np.concatenate([y_hand_tr, y_org_tr])
    
    X_test = np.concatenate([X_hand_te, X_org_te])
    y_test = np.concatenate([y_hand_te, y_org_te])

    print(f"  - Final Train Size: {X_train.shape}, Test Size: {X_test.shape}")
    return X_train, X_test, y_train, y_test

In [39]:
# ==========================================
# 4. 실험 실행 함수 (💡 파이프라인 결합 로직 추가)
# ==========================================
def run_experiment(scenario_name, X_train, X_test, y_train, y_test):
    print(f"\n{'='*60}")
    print(f"▶ Scenario: {scenario_name}")
    print(f"{'='*60}")

    # 학습은 Classifier 단독으로 진행 
    # (이미 X_train에 전처리가 완료되어 있으므로 Pipeline을 쓰면 중복 처리됨)
    param_dist = {
        "hidden_layer_sizes": [(100,), (150,), (100,100), (150,100), (200,100)],
        "alpha": [1e-5, 1e-4, 1e-3, 1e-2],
        "learning_rate_init": [0.0005, 0.001, 0.002, 0.005],
        "activation": ["relu", "tanh"],
        "batch_size": [64, 128, 256]
    }

    print("  - Starting Halving Random Search (MLP Only)...")
    
    mlp = MLPClassifier(
        random_state=42,
        solver="adam",
        max_iter=300,
        early_stopping=True
    )

    halving_search = HalvingRandomSearchCV(
        estimator=mlp,
        param_distributions=param_dist,
        factor=3,           # 라운드마다 후보 1/3로 줄임
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1,
        random_state=42,
    )
    
    # 🕒 [시간 측정 시작]
    start_time = time.time()

    print("search 객체 타입:", type(halving_search))
    halving_search.fit(X_train, y_train)
    
    # 🕒 [시간 측정 종료]
    end_time = time.time()
    training_time = end_time - start_time
    print(f"  🕒 Training Time: {training_time:.2f} seconds")

    # 최적 MLP 모델
    best_mlp_model = halving_search.best_estimator_
    print(f"\n  ★ Best Parameters: {halving_search.best_params_}")

    # 평가
    y_pred = best_mlp_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"  - Final Test Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))

    # 파이프라인 조립 및 저장
    full_pipeline = Pipeline([
        ('prep', CustomImagePreprocess()), 
        ('mlp', best_mlp_model)
    ])

    save_filename = f"pipeline_model_{scenario_name}.joblib"
    joblib.dump(full_pipeline, save_filename)
    print(f"  >> [Saved] Pipeline saved to '{save_filename}'")

    return acc, training_time

In [10]:
# ==========================================
# 5. Main 실행
# ==========================================
# (실제 파일 경로로 수정하세요)
hand_file = 'mnist_final_balanced.npz'
org_file = 'mnist_sklearn_40000.npz'

# 1배, 2배, 4배 실험 반복
factors = [1, 2, 4]

for factor in factors:
    scenario_name = f"HandAug_{factor}x_Preprocessed"
    X_tr, X_te, y_tr, y_te = load_preprocess_and_augment(hand_file, org_file, augment_factor=factor)
    
    if X_tr is not None:
        run_experiment(scenario_name, X_tr, X_te, y_tr, y_te)


[Data Prep] Loading & Preprocessing... Augment Factor: 1x
  - Preprocessing Handmade data...
  - Preprocessing Original MNIST data...
  - Final Train Size: (40896, 784), Test Size: (10224, 784)

▶ Scenario: HandAug_1x_Preprocessed
  - Starting Halving Random Search (MLP Only)...
search 객체 타입: <class 'sklearn.model_selection._search_successive_halving.HalvingRandomSearchCV'>
n_iterations: 6
n_required_iterations: 6
n_possible_iterations: 6
min_resources_: 60
max_resources_: 40896
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 480
n_resources: 60
Fitting 3 folds for each of 480 candidates, totalling 1440 fits


C:\Users\USER\ML\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 480 is smaller than n_iter=681. Running 480 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


----------
iter: 1
n_candidates: 160
n_resources: 180
Fitting 3 folds for each of 160 candidates, totalling 480 fits
----------
iter: 2
n_candidates: 54
n_resources: 540
Fitting 3 folds for each of 54 candidates, totalling 162 fits
----------
iter: 3
n_candidates: 18
n_resources: 1620
Fitting 3 folds for each of 18 candidates, totalling 54 fits
----------
iter: 4
n_candidates: 6
n_resources: 4860
Fitting 3 folds for each of 6 candidates, totalling 18 fits
----------
iter: 5
n_candidates: 2
n_resources: 14580
Fitting 3 folds for each of 2 candidates, totalling 6 fits
  🕒 Training Time: 832.21 seconds

  ★ Best Parameters: {'learning_rate_init': 0.005, 'hidden_layer_sizes': (150, 100), 'batch_size': 64, 'alpha': 1e-05, 'activation': 'relu'}
  - Final Test Accuracy: 0.9597
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1022
           1       0.93      0.98      0.96      1022
           2       0.98      0.94      0.96      1022
  

C:\Users\USER\ML\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 480 is smaller than n_iter=829. Running 480 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


----------
iter: 1
n_candidates: 160
n_resources: 180
Fitting 3 folds for each of 160 candidates, totalling 480 fits
----------
iter: 2
n_candidates: 54
n_resources: 540
Fitting 3 folds for each of 54 candidates, totalling 162 fits
----------
iter: 3
n_candidates: 18
n_resources: 1620
Fitting 3 folds for each of 18 candidates, totalling 54 fits
----------
iter: 4
n_candidates: 6
n_resources: 4860
Fitting 3 folds for each of 6 candidates, totalling 18 fits
----------
iter: 5
n_candidates: 2
n_resources: 14580
Fitting 3 folds for each of 2 candidates, totalling 6 fits
  🕒 Training Time: 1324.31 seconds

  ★ Best Parameters: {'learning_rate_init': 0.005, 'hidden_layer_sizes': (150, 100), 'batch_size': 64, 'alpha': 0.0001, 'activation': 'relu'}
  - Final Test Accuracy: 0.9594
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1022
           1       0.94      0.98      0.96      1022
           2       0.98      0.93      0.95      1022


C:\Users\USER\ML\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 480 is smaller than n_iter=1126. Running 480 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


----------
iter: 1
n_candidates: 160
n_resources: 180
Fitting 3 folds for each of 160 candidates, totalling 480 fits
----------
iter: 2
n_candidates: 54
n_resources: 540
Fitting 3 folds for each of 54 candidates, totalling 162 fits
----------
iter: 3
n_candidates: 18
n_resources: 1620
Fitting 3 folds for each of 18 candidates, totalling 54 fits
----------
iter: 4
n_candidates: 6
n_resources: 4860
Fitting 3 folds for each of 6 candidates, totalling 18 fits
----------
iter: 5
n_candidates: 2
n_resources: 14580
Fitting 3 folds for each of 2 candidates, totalling 6 fits
  🕒 Training Time: 296.47 seconds

  ★ Best Parameters: {'learning_rate_init': 0.005, 'hidden_layer_sizes': (150, 100), 'batch_size': 256, 'alpha': 0.01, 'activation': 'relu'}
  - Final Test Accuracy: 0.9645
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      1022
           1       0.95      0.98      0.96      1022
           2       0.98      0.95      0.96      1022
  

# RandomizedSearch바꾸고

In [15]:
from sklearn.model_selection import RandomizedSearchCV

# ==========================================
# 4. 실험 실행 함수 (💡 파이프라인 결합 로직 추가)
# ==========================================
def run_experiment(scenario_name, X_train, X_test, y_train, y_test):
    print(f"\n{'='*60}")
    print(f"▶ Scenario: {scenario_name}")
    print(f"{'='*60}")

    # 학습은 Classifier 단독으로 진행 
    # (이미 X_train에 전처리가 완료되어 있으므로 Pipeline을 쓰면 중복 처리됨)
    param_dist = {
        "hidden_layer_sizes": [(100,), (150,), (100,100), (150,100), (200,100)],
        "alpha": [1e-5, 1e-4, 1e-3, 1e-2],
        "learning_rate_init": [0.0005, 0.001, 0.002, 0.005],
        "activation": ["relu", "tanh"],
        "batch_size": [64, 128, 256],
        "solver": ["adam"],          # solver는 거의 adam이 안정적
        "max_iter": [200],           # RandomizedSearchCV에서는 고정 가능
        "early_stopping": [True],    # 과적합 방지
    }
    
    print("  - Starting Randomized Search (MLP Only)...")

    # --------------------------
    # 2) 기본 MLP 설정
    # --------------------------
    mlp = MLPClassifier(
        random_state=42
    )

    # --------------------------
    # 3) Randomized SearchCV
    # --------------------------
    random_search = RandomizedSearchCV(
        estimator=mlp,
        param_distributions=param_dist,
        n_iter=15,              
        cv=2,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )

    # ---- 학습 시간 측정 ----
    start_time = time.time()
    random_search.fit(X_train, y_train)
    training_time = time.time() - start_time
    print(f"\n  🕒 Training Time: {training_time:.2f} seconds")

    # --------------------------
    # 4) 최적 모델
    # --------------------------
    best_mlp_model = random_search.best_estimator_
    print(f"\n  ★ Best Parameters: {random_search.best_params_}")

    # --------------------------
    # 5) 성능 평가
    # --------------------------
    y_pred = best_mlp_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n  - Final Test Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))

    # --------------------------
    # 6) 최종 파이프라인 + 저장
    # --------------------------
    full_pipeline = Pipeline([
        ('prep', CustomImagePreprocess()),
        ('mlp', best_mlp_model)
    ])

    save_filename = f"mlp_{scenario_name}.joblib"
    joblib.dump(full_pipeline, save_filename)
    print(f"  >> [Saved] Pipeline saved to '{save_filename}'")

    return acc, training_time

In [16]:
# ==========================================
# 5. Main 실행
# ==========================================
# (실제 파일 경로로 수정하세요)
hand_file = 'mnist_final_balanced.npz'
org_file = 'mnist_sklearn_40000.npz'

# 1배, 2배, 4배 실험 반복
factors = [1, 2, 4]

for factor in factors:
    scenario_name = f"HandAug_{factor}x"
    X_tr, X_te, y_tr, y_te = load_preprocess_and_augment(hand_file, org_file, augment_factor=factor)
    
    if X_tr is not None:
        run_experiment(scenario_name, X_tr, X_te, y_tr, y_te)


[Data Prep] Loading & Preprocessing... Augment Factor: 1x
  - Preprocessing Handmade data...
  - Preprocessing Original MNIST data...
  - Final Train Size: (40896, 784), Test Size: (10224, 784)

▶ Scenario: HandAug_1x
  - Starting Randomized Search (MLP Only)...
Fitting 2 folds for each of 15 candidates, totalling 30 fits

  🕒 Training Time: 4281.01 seconds

  ★ Best Parameters: {'solver': 'adam', 'max_iter': 200, 'learning_rate_init': 0.001, 'hidden_layer_sizes': (200, 100), 'early_stopping': True, 'batch_size': 64, 'alpha': 0.0001, 'activation': 'relu'}

  - Final Test Accuracy: 0.9672
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1022
           1       0.97      0.98      0.97      1022
           2       0.97      0.96      0.97      1022
           3       0.96      0.97      0.97      1023
           4       0.95      0.98      0.97      1023
           5       0.97      0.96      0.97      1022
           6       0.97  

In [30]:
from joblib import load
import json

# 1️⃣ 모델 불러오기
model_path = "mlp_HandAug_1x.joblib"
best_mlp1= load(model_path)
print(f"✅ 모델 로드 완료: {model_path}")
model_path = "mlp_HandAug_2x.joblib"
best_mlp2= load(model_path)
print(f"✅ 모델 로드 완료: {model_path}")
model_path = "mlp_HandAug_4x.joblib"
best_mlp4= load(model_path)
print(f"✅ 모델 로드 완료: {model_path}")

✅ 모델 로드 완료: mlp_HandAug_4x.joblib


In [35]:
from joblib import load
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report

# 1️⃣ 모델 불러오기
models = {}
for name in ["HandAug_1x", "HandAug_2x", "HandAug_4x"]:
    path = f"mlp_{name}.joblib"
    models[name] = load(path)
    print(f"✅ 모델 로드 완료: {path}")

# 2️⃣ handmade 테스트 npz 로드
d = np.load("SMNIST_test_25.npz", allow_pickle=True)
print("파일 키:", d.keys())

Xh = d["img"]    # (N, 28, 28) 형태
yh = d["label"]  # (N,)

# label을 정수로 통일
yh = np.array([int(v) for v in yh])

# Xh를 (N, 784) 로 변환 + float32
if Xh.ndim == 3 and Xh.shape[1:] == (28, 28):
    Xh_flat = Xh.reshape(len(Xh), -1).astype(np.float32)
else:
    Xh_flat = Xh.astype(np.float32)

print("Xh_flat shape:", Xh_flat.shape)
print("yh shape     :", yh.shape)

# 3️⃣ 각 모델별 성능 비교
results = {}

for name, model in models.items():
    print("\n" + "="*60)
    print(f"🔍 모델: {name}")
    print("="*60)

    # 파이프라인이라 Xh_flat만 넣으면 내부에서 전처리 + 예측까지 수행
    y_pred = model.predict(Xh_flat)

    acc = accuracy_score(yh, y_pred)
    results[name] = acc

    print(f"🧪 Handmade Test Accuracy ({name}): {acc:.4f}")
    print("\n📄 Classification Report")
    print(classification_report(yh, y_pred, digits=4, zero_division=0))

# 4️⃣ 최종 요약 출력
print("\n\n===== 📊 모델별 Handmade Test Accuracy 요약 =====")
for name, acc in results.items():
    print(f"{name:15s} : {acc:.4f}")


✅ 모델 로드 완료: mlp_HandAug_1x.joblib
✅ 모델 로드 완료: mlp_HandAug_2x.joblib
✅ 모델 로드 완료: mlp_HandAug_4x.joblib
파일 키: KeysView(NpzFile 'SMNIST_test_25.npz' with keys: img, label)
Xh_flat shape: (2500, 784)
yh shape     : (2500,)

🔍 모델: HandAug_1x
🧪 Handmade Test Accuracy (HandAug_1x): 0.7556

📄 Classification Report
              precision    recall  f1-score   support

           0     0.6462    0.5480    0.5931       250
           1     0.6751    0.6400    0.6571       250
           2     0.7166    0.7080    0.7123       250
           3     0.8255    0.7000    0.7576       250
           4     0.8387    0.8320    0.8353       250
           5     0.8023    0.8280    0.8150       250
           6     0.6046    0.9480    0.7383       250
           7     0.8346    0.8680    0.8510       250
           8     0.8502    0.7040    0.7702       250
           9     0.8590    0.7800    0.8176       250

    accuracy                         0.7556      2500
   macro avg     0.7653    0.7556    0.754

In [37]:
from joblib import load
import json

# 1️⃣ 모델 불러오기
model_path = "mlp_HandAug_4x.joblib"
best_mlp= load(model_path)
print(f"✅ 모델 로드 완료: {model_path}")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1) handmade npz 로드 (파일명/경로는 상황에 맞게)
d = np.load("mnist23_1-32\combined_mnist_1to32.npz.")
print(d.keys())
Xh = d["img"]   # (N, 28, 28) 또는 (N, 784)
yh = d["label"] # (N,)

yh = np.array([int(v) for v in yh])

model=best_mlp
y_pred_h = model.predict(Xh)

# 3) 성능 지표
print("🧪 Handmade Test | Accuracy:", accuracy_score(yh, y_pred_h))
print("\n📄 Classification Report (Handmade Test)")
print(classification_report(yh, y_pred_h, digits=4, zero_division=0))

✅ 모델 로드 완료: mlp_HandAug_4x.joblib
KeysView(NpzFile 'mnist23_1-32\\combined_mnist_1to32.npz.' with keys: img, label)
🧪 Handmade Test | Accuracy: 0.8437894736842105

📄 Classification Report (Handmade Test)
              precision    recall  f1-score   support

           0     0.8666    0.8684    0.8675       950
           1     0.9000    0.8432    0.8707       950
           2     0.8774    0.8516    0.8643       950
           3     0.8275    0.8179    0.8227       950
           4     0.8533    0.8632    0.8582       950
           5     0.6903    0.8516    0.7625       950
           6     0.8280    0.8663    0.8467       950
           7     0.8944    0.8116    0.8510       950
           8     0.8714    0.8200    0.8449       950
           9     0.8775    0.8442    0.8605       950

    accuracy                         0.8438      9500
   macro avg     0.8486    0.8438    0.8449      9500
weighted avg     0.8486    0.8438    0.8449      9500

